### Porcentaje de hogares que destinan más del 10% de su gasto a transporte público en la Ciudad de México

**Indicador:** Porcentaje de hogares que destinan más del 10% de su ingreso corriente a transporte público | **Fuente:** INEGI, Encuesta Nacional de Ingresos y Gastos de los Hogares (ENIGH) 2016, 2018, 2020, 2022, 2024, microdatos de concentrado de hogar y gastos | **Universo:** Hogares con ingreso corriente positivo en la Ciudad de México (cve_ent==09)

**Método:** Estimación con diseño muestral complejo (upm, estrato, factor de expansión) sobre microdatos ENIGH. El gasto trimestral en transporte público se identifica por claves de gasto de persona y hogar (catálogo distinto a partir de 2024) y se marca el hogar como afectado cuando ese gasto supera el 10% de su ingreso corriente. La prospectiva a 2047 usa la TVMA (tasa de variación media anual) histórica 2020-2024 combinada con una meta absoluta para el escenario deseable.

In [1]:
rm(list = ls())

if (!require("pacman")) install.packages("pacman")
pacman::p_load(tidyverse, srvyr)
options(survey.lonely.psu = "certainty")

Cargando paquete requerido: pacman



In [2]:
base_dir <- "D:/Descargas/data"
years    <- c("2016", "2018", "2020", "2022", "2024")

In [3]:
#' Estima el porcentaje de hogares que destinan más del 10% de su ingreso corriente a transporte público
estimacion_10pct <- function(year) {
  message("Procesando: ", year)

  leer <- function(nombre) {
    read_csv(file.path(base_dir, sprintf("ns_%s_%s.csv", nombre, year)),
             col_types = cols(.default = col_character()))
  }

  hogares <- leer("concentradohogar") %>%
    mutate(ing_cor = as.numeric(ing_cor),
           factor  = as.numeric(factor),
           cve_ent = substr(str_pad(folioviv, 10, pad = "0"), 1, 2))

  gp <- leer("gastospersona") %>% mutate(gasto_tri = as.numeric(gasto_tri))
  gh <- leer("gastoshogar")   %>% mutate(gasto_tri = as.numeric(gasto_tri))

  claves_tp <- if (as.integer(year) >= 2024) {
    c("073111", "073112", "073121", "073122", "073123",
      "073211", "073212", "073213", "073214", "073215",
      "073221", "073222", "073230", "073600")
  } else {
    c("B001", "B002", "B003", "B004", "B005", "B006")
  }

  gasto_per <- gp %>%
    filter(clave %in% claves_tp) %>%
    group_by(folioviv, foliohog) %>%
    summarise(g_per = sum(gasto_tri, na.rm = TRUE), .groups = "drop")

  gasto_hog <- gh %>%
    filter(clave %in% claves_tp) %>%
    group_by(folioviv, foliohog) %>%
    summarise(g_hog = sum(gasto_tri, na.rm = TRUE), .groups = "drop")

  hogares <- hogares %>%
    left_join(gasto_per, by = c("folioviv", "foliohog")) %>%
    left_join(gasto_hog, by = c("folioviv", "foliohog")) %>%
    mutate(
      g_per     = replace_na(g_per, 0),
      g_hog     = replace_na(g_hog, 0),
      gasto_tp  = g_per + g_hog,
      diezpct   = if_else(ing_cor > 0 & (gasto_tp / ing_cor) > 0.10, 1, 0)
    )

  dm <- hogares %>%
    as_survey_design(ids = upm, strata = est_dis, weights = factor, nest = TRUE)

  dm %>%
    filter(cve_ent == "09") %>%
    group_by(diezpct) %>%
    summarise(
      hogares    = survey_total(vartype = "cv"),
      porcentaje = survey_prop(vartype = "cv")
    ) %>%
    mutate(porcentaje = round(porcentaje * 100, 2),
           year       = year) %>%
    filter(diezpct == 1)
}

In [4]:
resultados <- map_dfr(years, estimacion_10pct)
resultados

Procesando: 2016



Procesando: 2018

Procesando: 2020

Procesando: 2022

Procesando: 2024



diezpct,hogares,hogares_cv,porcentaje,porcentaje_cv,year
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
1,331752,0.07502898,11.74,0.07402754,2016
1,381966,0.06938696,13.75,0.06654310,2018
1,169182,0.09818059,6.19,0.09659305,2020
1,252867,0.08180108,8.46,0.08076311,2022
1,162643,0.10712784,5.28,0.10580288,2024


In [5]:
anio_base  <- 2024
linea_base <- resultados$porcentaje[resultados$year == "2024"]

tvma <- (resultados$porcentaje[resultados$year == "2024"] /
         resultados$porcentaje[resultados$year == "2020"]) ^ (1 / (2024 - 2020)) - 1

tvma

[1] -0.03897249

In [12]:
anios_proy <- (anio_base + 1):2047
n_periodos <- anios_proy - anio_base

delta_inercial      <- tvma
delta_deseable      <- (2.12 / linea_base) ^ (1 / (2047 - anio_base)) - 1
delta_transformador <- delta_deseable * 1.5

cat(sprintf("Delta inercial:      %.4f (%.2f%% anual)\n", delta_inercial, delta_inercial * 100))
cat(sprintf("Delta deseable:      %.4f (%.2f%% anual)\n", delta_deseable, delta_deseable * 100))
cat(sprintf("Delta transformador: %.4f (%.2f%% anual)\n", delta_transformador, delta_transformador * 100))

Delta inercial:      -0.0390 (-3.90% anual)


Delta deseable:      -0.0389 (-3.89% anual)
Delta transformador: -0.0583 (-5.83% anual)


In [13]:
escenarios <- tibble(
  year          = anios_proy,
  inercial      = linea_base * (1 + delta_inercial) ^ n_periodos,
  deseable      = linea_base * (1 + delta_deseable) ^ n_periodos,
  transformador = linea_base * (1 + delta_transformador) ^ n_periodos
)

observados <- resultados %>%
  mutate(year = as.numeric(year)) %>%
  select(year, observado = porcentaje)

tabla_final <- escenarios %>%
  full_join(observados, by = "year") %>%
  mutate(
    inercial      = if_else(!is.na(observado), observado, inercial),
    deseable      = if_else(!is.na(observado), observado, deseable),
    transformador = if_else(!is.na(observado), observado, transformador)
  ) %>%
  select(periodo = year, inercial, deseable, transformador) %>%
  arrange(periodo) %>%
  mutate(across(c(inercial, deseable, transformador), ~ round(., 2)))

tabla_final

periodo,inercial,deseable,transformador
<dbl>,<dbl>,<dbl>,<dbl>
2016,11.74,11.74,11.74
2018,13.75,13.75,13.75
2020,6.19,6.19,6.19
2022,8.46,8.46,8.46
2024,5.28,5.28,5.28
2025,5.07,5.07,4.97
2026,4.88,4.88,4.68
2027,4.69,4.69,4.41
2028,4.50,4.51,4.15


In [ ]:
out_path <- "hogares_que_destinan_mas_10pct_2024.csv"
write.csv(tabla_final, file = out_path, row.names = FALSE, fileEncoding = "UTF-8")
cat(sprintf("Guardado en: %s\n", out_path))

Guardado en: METAS_hogares_que_destinan_mas_10pct_2024.csv
